# Set up library

In [ ]:
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.callbacks import ModelCheckpoint,  EarlyStopping
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, Flatten, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
import h5py
from sklearn.utils.class_weight import compute_class_weight
# from autolrtuner import AutoLRTuner
import os
# from audiomentations import Compose, AddGaussianNoise, Gain





# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Main

### encoding labels

In [ ]:
HDF5_file_path = "/content/drive/MyDrive/Qeej_Hmong_Dataset/HDF5/Qeej_Hmong_Features.h5"
y = None
if os.path.exists(HDF5_file_path):
      print("Retrieving data from drive with path:", HDF5_file_path)
      with h5py.File(HDF5_file_path, "r") as f:
        y = f["labels"][:]
        y = [name.decode('utf-8') for name in y]

le = LabelEncoder()
le.fit_transform(y)
class_names = le.classes_
print("check class_names", class_names)

Retrieving data from drive with path: /content/drive/MyDrive/Qeej_Hmong_Dataset/HDF5/Qeej_Hmong_Features.h5
check class_names ['P1' 'P1_P2' 'P1_P2_P3' 'P1_P2_P3_T1' 'P1_P2_P3_T1_T2'
 'P1_P2_P3_T1_T2_T3' 'P1_P2_P3_T1_T3' 'P1_P2_P3_T2' 'P1_P2_P3_T2_T3'
 'P1_P2_P3_T3' 'P1_P2_T1' 'P1_P2_T1_T2' 'P1_P2_T1_T2_T3' 'P1_P2_T1_T3'
 'P1_P2_T2' 'P1_P2_T2_T3' 'P1_P2_T3' 'P1_P3' 'P1_P3_T1' 'P1_P3_T1_T2'
 'P1_P3_T1_T2_T3' 'P1_P3_T1_T3' 'P1_P3_T2' 'P1_P3_T2_T3' 'P1_P3_T3'
 'P1_T1' 'P1_T1_T2' 'P1_T1_T2_T3' 'P1_T1_T3' 'P1_T2' 'P1_T2_T3' 'P1_T3'
 'drone' 'drone_P2' 'drone_P2_P3' 'drone_P2_P3_T1' 'drone_P2_P3_T1_T2'
 'drone_P2_P3_T1_T2_T3' 'drone_P2_P3_T1_T3' 'drone_P2_P3_T2'
 'drone_P2_P3_T2_T3' 'drone_P2_P3_T3' 'drone_P2_T1' 'drone_P2_T1_T2'
 'drone_P2_T1_T2_T3' 'drone_P2_T1_T3' 'drone_P2_T2' 'drone_P2_T2_T3'
 'drone_P2_T3' 'drone_P3' 'drone_P3_T1' 'drone_P3_T1_T2'
 'drone_P3_T1_T2_T3' 'drone_P3_T1_T3' 'drone_P3_T2' 'drone_P3_T2_T3'
 'drone_P3_T3' 'drone_T1' 'drone_T1_T2' 'drone_T1_T2_T3' 'drone_T1_T3'
 

In [ ]:
Model_Path = "/content/drive/MyDrive/Qeej_Hmong_Model_Parameters/CheckPoints/model_latest (1).keras"
model = load_model(Model_Path)

In [ ]:
def extract_dft(window):
  eps = 1e-10
  spectrum = np.abs(np.fft.rfft(window))
  spectrum = spectrum / np.max(spectrum + eps)
  return spectrum

def suppress_low_amplitudes(spec, threshold_db=50):
  I_threshold = 10 ** (-threshold_db / 20)
  spec[spec < I_threshold] *= 0.2
  return spec
def get_spectrum_of_window(window):

  spectrum = extract_dft(window)

  #preprocessing
  standard_spectrum = suppress_low_amplitudes(spectrum)

  return standard_spectrum

In [ ]:
def get_spectrograms_of_audio(audio, samples_per_window):
        X = []
        # preparing data
        for i in np.arange(0, len(audio), samples_per_window):
            start_sample = i
            end_sample = i + samples_per_window
            window = audio[start_sample:end_sample]
            spectrum = get_spectrum_of_window(window)

            X.append(spectrum)


        return np.array(X)


In [ ]:
import json
import unicodedata
file_path = "/content/drive/MyDrive/Real Data/Result/results_from_real_data.json"
result_dict = {}
def assess_audio(
        audio_path,
        sample_rate=48000,
        window_size=0.01
    ):


        #load audio
        audio, sr = librosa.load(audio_path, sr=sample_rate, mono=True)

        print("check len audio: ", len(audio))

        samples_per_window = int(sample_rate * window_size)
        print("samples per window: ", samples_per_window)

        #get features X
        X = get_spectrograms_of_audio(audio, samples_per_window)

        # predictions results
        y_pred = model.predict(X[..., np.newaxis])
        y_pred = np.argmax(y_pred, axis=1)
        y_to_str = le.inverse_transform(y_pred)

        #check the most appear
        unique, counts = np.unique(y_to_str, return_counts=True)

        max_label = unique[np.argmax(counts)]

        audio_path = unicodedata.normalize("NFC", audio_path)

        if audio_path not in result_dict:
            result_dict[audio_path] = max_label







## assess all audio

In [ ]:
 # read data from drive
audio_data_set_path = "/content/drive/MyDrive/Real Data/Data"

if os.path.exists(audio_data_set_path):
  for root, dirs, files in os.walk(audio_data_set_path):
    for file in files:
      if file.endswith(".wav"):
        audio_file_path = os.path.join(root, file)
        assess_audio(audio_file_path)

        with open(file_path, "w", encoding="utf-8") as f:
          json.dump(result_dict, f, ensure_ascii=False, indent=4)




else:
  print("Cant find folder at /content/drive/MyDrive/Qeej_Hmong_Dataset/Data/audio")


check len audio:  68160
samples per window:  480
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 121ms/step
check len audio:  73920
samples per window:  480
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 128ms/step
check len audio:  66240
samples per window:  480
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
check len audio:  54720
samples per window:  480
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 164ms/step
check len audio:  60480
samples per window:  480
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
check len audio:  83520
samples per window:  480
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
check len audio:  66240
samples per window:  480
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
check len audio:  66240
samples per window:  480
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
check len audio:  50880
samples per window:  480
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
check len audio:  62400
samples per window:  480
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
check len audio:  48960
samples per window:  480
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
check len audio:  45120
samples per wi